# Kidney Disease Prediction
Production notebook for MediPredictAI

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score


## Load Dataset

In [2]:
df=pd.read_csv('../datasets/kidney/kidney_disease.csv')
df.head()

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


## Explore Dataset

In [3]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df['classification'].value_counts())

(400, 26)
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             391 non-null    float64
 2   bp              388 non-null    float64
 3   sg              353 non-null    float64
 4   al              354 non-null    float64
 5   su              351 non-null    float64
 6   rbc             248 non-null    str    
 7   pc              335 non-null    str    
 8   pcc             396 non-null    str    
 9   ba              396 non-null    str    
 10  bgr             356 non-null    float64
 11  bu              381 non-null    float64
 12  sc              383 non-null    float64
 13  sod             313 non-null    float64
 14  pot             312 non-null    float64
 15  hemo            348 non-null    float64
 16  pcv             330 non-null    str    
 17  wc              295 non-null    str 

## Clean Dataset

In [4]:
df=df.replace('?',np.nan)
df=df.replace('\t?',np.nan)
df=df.replace('\tyes','yes')
df=df.replace('\tno','no')
df=df.replace('ckd\t','ckd')

if 'id' in df.columns:
    df=df.drop('id',axis=1)


## Encode Target

In [5]:
df['classification']=df['classification'].replace({'ckd':1,'notckd':0}).astype(int)

## Separate Features

In [6]:
X=df.drop('classification',axis=1)
y=df['classification']

## Handle Missing Values

In [7]:
num_cols=X.select_dtypes(include=['int64','float64']).columns
cat_cols=X.select_dtypes(include=['object']).columns

num_imputer=SimpleImputer(strategy='median')
cat_imputer=SimpleImputer(strategy='most_frequent')

X[num_cols]=num_imputer.fit_transform(X[num_cols])
X[cat_cols]=cat_imputer.fit_transform(X[cat_cols])


C:\Users\CodeWithPranav\AppData\Local\Temp\ipykernel_3496\2734307727.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols=X.select_dtypes(include=['object']).columns


## One Hot Encoding

In [8]:
X=pd.get_dummies(X,columns=cat_cols,drop_first=False)
feature_columns=X.columns.tolist()

## Train Test Split

In [9]:
X_train,X_test,y_train,y_test=train_test_split(
X,y,test_size=0.2,random_state=42,stratify=y)

## Feature Scaling

In [10]:
scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)


## Train Models

In [11]:
models={
'Logistic Regression':LogisticRegression(max_iter=1000),
'Decision Tree':DecisionTreeClassifier(random_state=42),
'Random Forest':RandomForestClassifier(random_state=42),
'KNN':KNeighborsClassifier(),
'SVM':SVC(probability=True),
'Naive Bayes':GaussianNB()
}

results=[]

for name,model in models.items():
    model.fit(X_train_scaled,y_train)
    pred=model.predict(X_test_scaled)
    results.append({
        'Model':name,
        'Accuracy':accuracy_score(y_test,pred),
        'Precision':precision_score(y_test,pred),
        'Recall':recall_score(y_test,pred),
        'F1 Score':f1_score(y_test,pred)
    })

results_df=pd.DataFrame(results).sort_values(by='Accuracy',ascending=False)
results_df


C:\Users\CodeWithPranav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Accuracy,Precision,Recall,F1 Score
2,Random Forest,1.0000,1.000000,1.00,1.000000
0,Logistic Regression,0.9875,1.000000,0.98,0.989899
1,Decision Tree,0.9875,1.000000,0.98,0.989899
4,SVM,0.9250,0.958333,0.92,0.938776
3,KNN,0.9000,0.888889,0.96,0.923077
5,Naive Bayes,0.9000,0.956522,0.88,0.916667


## Save Best Model

In [12]:
best_model_name=results_df.iloc[0]['Model']
best_model=models[best_model_name]
print(best_model_name)

MODEL_DIR=Path('../trained_models')
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(best_model,MODEL_DIR/'kidney_model.pkl')
joblib.dump(scaler,MODEL_DIR/'kidney_scaler.pkl')
joblib.dump(feature_columns,MODEL_DIR/'kidney_columns.pkl')

print('Saved Successfully')


Random Forest
Saved Successfully


## Test Saved Model

In [13]:
model=joblib.load('../trained_models/kidney_model.pkl')
scaler=joblib.load('../trained_models/kidney_scaler.pkl')
columns=joblib.load('../trained_models/kidney_columns.pkl')

sample=X.iloc[[0]].copy()
sample=sample.reindex(columns=columns,fill_value=0)
sample=scaler.transform(sample)

print(model.predict(sample))
print(model.predict_proba(sample))


[1]
[[0.09 0.91]]
